In [5]:
from datasets import load_dataset
import pandas as pd

In [6]:

dataset = load_dataset("jfleg")   # public dataset on Hugging Face
dataset = dataset['validation']    # small subset for quick test



In [7]:
dataset

Dataset({
    features: ['sentence', 'corrections'],
    num_rows: 755
})

In [8]:
dataset = dataset.rename_column("sentence", "input_text")
dataset = dataset.rename_column("corrections", "target_text")


dataset = dataset.train_test_split(test_size=0.2)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(" Dataset loaded.")
print(train_dataset[0])

 Dataset loaded.
{'input_text': 'The same will happen to taxation on cars and gasoline . ', 'target_text': ['The same will happen to taxation on cars and gasoline . ', 'The same will happen to taxes on cars and gasoline . ', 'The same will happen to taxation on cars and gasoline . ', 'The same will happen to taxation on cars and gasoline . ']}


In [9]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [10]:
#  Step 1: Handle multiple corrections
def flatten_targets(example):
    # If target_text is a list (multiple corrections), pick the first one
    if isinstance(example["target_text"], list):
        example["target_text"] = example["target_text"][0]
    return example

train_dataset = train_dataset.map(flatten_targets)
test_dataset = test_dataset.map(flatten_targets)

# Quick sanity check
print(" Sample after flattening:")
print(train_dataset[0])


Map:   0%|          | 0/604 [00:00<?, ? examples/s]

Map:   0%|          | 0/151 [00:00<?, ? examples/s]

 Sample after flattening:
{'input_text': 'The same will happen to taxation on cars and gasoline . ', 'target_text': 'The same will happen to taxation on cars and gasoline . '}


In [11]:

#  Step 2: Preprocessing (tokenization)
max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    # Prefix with "fix:" to indicate correction task
    inputs = ["fix: " + str(text) for text in examples["input_text"]]
    targets = [str(text) for text in examples["target_text"]]

    # Tokenize source
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    # Tokenize target
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=max_target_length,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

#  Tokenize datasets
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_test = test_dataset.map(preprocess_function, batched=True, remove_columns=test_dataset.column_names)

print(" Tokenization complete. Example:")
print({k: v[:5] for k, v in tokenized_train[0].items()})


Map:   0%|          | 0/604 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/151 [00:00<?, ? examples/s]

 Tokenization complete. Example:
{'input_ids': [2210, 10, 37, 337, 56], 'attention_mask': [1, 1, 1, 1, 1], 'labels': [37, 337, 56, 1837, 12]}


In [12]:

#  Step 3: Train the model
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    fp16=True,
    logging_steps=10,
    report_to="none"   #  This disables wandb and all online loggers
)




In [13]:
import os
os.environ["WANDB_DISABLED"] = "true"

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

# Train the model
trainer.train()

Step,Training Loss
10,6.893500
20,0.838900
30,0.398000
40,0.369300
50,0.300000
60,0.327400
70,0.198600
80,0.192800
90,0.224000
100,0.181700


TrainOutput(global_step=453, training_loss=0.32050097988667603, metrics={'train_runtime': 58.0659, 'train_samples_per_second': 31.206, 'train_steps_per_second': 7.801, 'total_flos': 61309836066816.0, 'train_loss': 0.32050097988667603, 'epoch': 3.0})

In [ ]:

#  Step 4: Save model
trainer.save_model("./grammar_correction_t5")
tokenizer.save_pretrained("./grammar_correction_t5")
print(" Model saved successfully at './grammar_correction_t5'")


✅ Model saved successfully at './grammar_correction_t5'


In [14]:
#  Step 5: Testing the model
from transformers import T5ForConditionalGeneration, T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("./grammar_correction_t5")
model = T5ForConditionalGeneration.from_pretrained("./grammar_correction_t5")

def correct_sentence(sentence):
    input_text = "fix: " + sentence
    inputs = tokenizer.encode(input_text, return_tensors="pt", max_length=128, truncation=True)
    outputs = model.generate(inputs, max_length=128, num_beams=5, early_stopping=True)
    corrected = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return corrected

# Test few examples
print("\n Model test results:")
print("Input: she eat food →", correct_sentence("she eat food"))
print("Input: she go to school yesterday →", correct_sentence("she go to school yesterday"))
print("Input: he not like apple →", correct_sentence("i no specific choice apple , i eat whatever you give"))


OSError: Can't load tokenizer for './grammar_correction_t5'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure './grammar_correction_t5' is the correct path to a directory containing all relevant files for a T5Tokenizer tokenizer.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy the saved model to Drive
!cp -r ./grammar_correction_t5 /content/drive/MyDrive/
print(" Model copied to Drive: /content/drive/MyDrive/grammar_correction_t5")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Model copied to Drive: /content/drive/MyDrive/grammar_correction_t5


In [ ]:
!ls -lh


total 8.0K
drwx------ 5 root root 4.0K Nov  7 09:36 drive
drwxr-xr-x 1 root root 4.0K Nov  5 14:33 sample_data


In [ ]:
!find /content -type d -name "grammar_correction_t5"
